In [70]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')

import os

print(os.listdir('/content/drive/MyDrive'))

import zipfile
zip_path = "/content/drive/MyDrive/Mammo-Bench/Preprocessed_Dataset.zip"
extract_path = "/content/"



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Snapchat-1429388362.jpg', 'IMG_20220322_142117_384.jpg', 'IMG_20220322_141947_872.jpg', 'IMG_20220225_150250_575.jpg', 'IMG_20220225_150248_041.jpg', 'IMG_20220225_150246_134.jpg', 'IMG_20220225_150243_901.jpg', 'IMG_20220102_210029_060.jpg', 'IMG_20211218_114122_132.jpg', 'IMG_20220322_142120_048.jpg', '208acc8c-1e2a-4238-8361-b4198da3a397.mp4', 'a8e4b022-64cb-4e36-9fc8-0374585235e4.mp4', '6374ff8d-04ab-49fb-8d69-db1ae2897525.jpg', 'IMG_1862.PNG', '968f8636-c90e-44c2-b5e6-6f027457a7f4.jpg', '6d8f8afa-4c08-4bd3-b39d-feb71816ec38.jpg', '9012aed0-5bba-4574-9856-68800c88bde9.jpg', '8d7dd70a-f569-4a62-8134-4321f1622b7c.jpg', '582184a6-b426-427c-8edf-d1b9150b5218.jpg', '21fbb7fe-0acf-462c-9136-6a25154e113f.jpg', '666a2d4a-48db-4324-b13b-a16cd26e951c.jpg', '288073dd-c1b2-4c47-977c-3866e6690a59.mp4', '8450b92b-a3b8-4579-ae27-e3edc55a2ed6.mp4', '0f7e3e35-99be-4f57-

In [71]:
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully")

Unzipped successfully


In [72]:
import pandas as pd
DATA_ROOT = "/content"
CSV_PATH = "/content/drive/MyDrive/Mammo-Bench/metadata.csv"

df = pd.read_csv(CSV_PATH)

sample_path = df.iloc[0]["preprocessed_image_path"]
full_path = os.path.join(DATA_ROOT, sample_path)

print(full_path)
print(os.path.exists(full_path))

/content/Preprocessed_Dataset/inbreast/inbreast_0.jpg
True


In [73]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [74]:
def load_resnet50_model(model_path, num_classes):
    model = models.resnet50(weights=None)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()

    return model

In [75]:
cancer_model_path = "/content/drive/MyDrive/Mammo-Bench/best_restnet50_5_mammobench.pth"
density_model_path = "/content/drive/MyDrive/Mammo-Bench/best_restnet50_density_mammobench.pth"
birads_model_path = "/content/drive/MyDrive/Mammo-Bench/best_restnet50_birads_mammobench.pth"

cancer_model = load_resnet50_model(cancer_model_path, num_classes=3)
density_model = load_resnet50_model(density_model_path, num_classes=4)
birads_model = load_resnet50_model(birads_model_path, num_classes=5)

cancer_classes = ["Normal", "Benign", "Malignant"]
density_classes = ["A", "B", "C", "D"]
birads_classes = [1,2,3,4,5]

In [76]:
def predict_image(model, image_path, class_names):
    image = Image.open(image_path).convert("RGB")
    image_tensor = image_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        predicted_index = torch.argmax(probabilities).item()

    predicted_class = class_names[predicted_index]
    confidence = probabilities[predicted_index].item()

    prob_dict = {
        class_names[i]: probabilities[i].item()
        for i in range(len(class_names))
    }

    return predicted_class, confidence, prob_dict

In [77]:
def calculate_risk_score(cancer_probs, predicted_density, birads):
    p_benign = cancer_probs["Benign"]
    p_malignant = cancer_probs["Malignant"]

    cnn_risk = (p_malignant * 100) + (p_benign * 30)

    density_risk_map = {
        "A": 25,
        "B": 50,
        "C": 75,
        "D": 100
    }

    birads_risk_map = {
        1: 0,
        2: 25,
        3: 50,
        4: 75,
        5: 100
    }

    density_risk = density_risk_map[predicted_density]
    birads_risk = birads_risk_map[birads]

    final_risk = (
        0.60 * cnn_risk +
        0.25 * birads_risk +
        0.15 * density_risk
    )

    final_risk = max(0, min(final_risk, 100))

    if final_risk >= 67:
        risk_level = "High Risk"
        msg = "The results indicate a higher level of risk. It is important to seek medical advice promptly to ensure appropriate evaluation and care."
    elif final_risk >= 34:
        risk_level = "Medium Risk"
        msg= "Some areas may need further review. A follow-up consultation with a healthcare professional is recommended."
    else:
        risk_level = "Low Risk"
        msg = "No major concerns are indicated at this time. Continue with regular check-ups and screenings."

    return {
        "cnn_risk": cnn_risk,
        "density_risk": density_risk,
        "birads_risk": birads_risk,
        "final_risk_score": final_risk,
        "risk_level": risk_level,
        "msg":msg
    }

In [78]:
def predict_cancer_risk(image_path):


    cancer_pred, cancer_conf, cancer_probs = predict_image(
        cancer_model,
        image_path,
        cancer_classes
    )


    density_pred, density_conf, density_probs = predict_image(
        density_model,
        image_path,
        density_classes
    )


    birads_pred, birads_conf, birads_probs = predict_image(
        birads_model,
        image_path,
        birads_classes
    )
    if cancer_pred == "Malignant":

        result = {
            "predicted_cancer_class": cancer_pred,
            #"cancer_confidence": round(cancer_conf * 100, 2),

           # "cancer_probabilities": {
              #  key: round(value * 100, 2)
              #  for key, value in cancer_probs.items()
            #},

            "predicted_density": density_pred,
           # "density_confidence": round(density_conf * 100, 2),

           # "density_probabilities": {
               # key: round(value * 100, 2)
               # for key, value in density_probs.items()
           # },

            "predicted_birads": birads_pred,
           # "birads_confidence": round(birads_conf * 100, 2),

           # "birads_probabilities": {
               # key: round(value * 100, 2)
             #   for key, value in birads_probs.items()
           # },

            "future_risk_score": None,

            "risk_level": "Not Applicable",

            "feedback": (
                "A malignant finding was detected. "
                "Future cancer risk estimation is not applicable "
                "because the case is already classified as cancer-suspicious. "
                "Please seek medical review."
            )
        }

        return result

    risk_result = calculate_risk_score(
        cancer_probs=cancer_probs,
        predicted_density=density_pred,
        birads=birads_pred
    )

    result = {

        "predicted_cancer_class": cancer_pred,
      #  "cancer_confidence": round(cancer_conf * 100, 2),

        #"cancer_probabilities": {
            #key: round(value * 100, 2)
            #for key, value in cancer_probs.items()
        #},

        "predicted_density": density_pred,
       # "density_confidence": round(density_conf * 100, 2),

        #"density_probabilities": {
            #key: round(value * 100, 2)
           # for key, value in density_probs.items()
       # },

        "predicted_birads": birads_pred,
       # "birads_confidence": round(birads_conf * 100, 2),

        #"birads_probabilities": {
          #  key: round(value * 100, 2)
           # for key, value in birads_probs.items()
      #  },


        "cnn_risk_score": round(risk_result["cnn_risk"], 2),

        "density_risk_score": round(
            risk_result["density_risk"], 2
        ),

        "birads_risk_score": round(
            risk_result["birads_risk"], 2
        ),

        "future_risk_score": round(
            risk_result["final_risk_score"], 2
        ),

        "risk_level": risk_result["risk_level"],

        "feedback": risk_result["msg"]
    }

    return result

In [79]:
df["full_image_path"] = df["preprocessed_image_path"].apply(
    lambda x: os.path.join(DATA_ROOT, x)
)

print(df['full_image_path'].iloc[0])

sample_image_path = df["full_image_path"].iloc[12]


result = predict_cancer_risk(sample_image_path)
print(result)

/content/Preprocessed_Dataset/inbreast/inbreast_0.jpg
{'predicted_cancer_class': 'Malignant', 'predicted_density': 'A', 'predicted_birads': 2, 'future_risk_score': None, 'risk_level': 'Not Applicable', 'feedback': 'A malignant finding was detected. Future cancer risk estimation is not applicable because the case is already classified as cancer-suspicious. Please seek medical review.'}


In [80]:
def predict_multiple_image_risk(image_paths):

    image_level_results = []

    # Predict each image one by one
    for image_path in image_paths:
        result = predict_cancer_risk(image_path)
        result["image_path"] = image_path
        image_level_results.append(result)

    # --------------------------------------------------
    # If any image is malignant, stop future risk scoring
    # --------------------------------------------------
    malignant_results = [
        result for result in image_level_results
        if result["predicted_cancer_class"] == "Malignant"
    ]
    predicted_classes = [
        result["predicted_cancer_class"]
        for result in image_level_results
    ]

    if "Malignant" in predicted_classes:
        final_predicted_class = "Malignant"

    elif "Benign" in predicted_classes:
        final_predicted_class = "Benign"

    else:
        final_predicted_class = "Normal"


    if len(malignant_results) > 0:
        return {
            "number_of_images": len(image_paths),
            "image_level_results": image_level_results,
            "status": "Malignant detected",
            "future_risk_score": None,
            "risk_level": "Not Applicable",
            "feedback": (
               "A malignant finding was detected in at least one image. "
                "Future cancer risk estimation is not applicable because the case "
                "is already classified as cancer-suspicious. Please seek medical review."
            )
        }

    # --------------------------------------------------
    # If no malignant image, calculate final future risk
    # --------------------------------------------------
    highest_cnn_risk = max(
        result["cnn_risk_score"] for result in image_level_results
    )

    highest_density_risk = max(
        result["density_risk_score"] for result in image_level_results
    )

    highest_birads_risk = max(
        result["birads_risk_score"] for result in image_level_results
    )

    final_future_risk_score = (
        0.60 * highest_cnn_risk +
        0.15 * highest_density_risk +
        0.25 * highest_birads_risk
    )

    final_future_risk_score = max(0, min(final_future_risk_score, 100))

    if final_future_risk_score >= 67:
        risk_level = "High Risk"
        feedback = (
            "The results indicate a higher future cancer risk. "
            "A medical review is strongly recommended."
        )
    elif final_future_risk_score >= 34:
        risk_level = "Medium Risk"
        feedback = (
            "The results indicate a moderate future cancer risk. "
            "A follow-up consultation or further review is recommended."
        )
    else:
        risk_level = "Low Risk"
        feedback = (
            "The results indicate a lower future cancer risk. "
            "Regular screening and medical follow-up are still recommended."
        )

    return {
        "number_of_images": len(image_paths),
        "image_level_results": image_level_results,
        "final_predicted_class": final_predicted_class,
        "highest_cnn_risk_score": round(highest_cnn_risk, 2),
        "highest_density_risk_score": round(highest_density_risk, 2),
        "highest_birads_risk_score": round(highest_birads_risk, 2),

        "future_risk_score": round(final_future_risk_score, 2),
        "risk_level": risk_level,
        "feedback": feedback
    }

In [82]:
df["full_image_path"] = df["preprocessed_image_path"].apply(
    lambda x: os.path.join(DATA_ROOT, x)
)

image_paths = [
    df["full_image_path"].iloc[1],
    df["full_image_path"].iloc[2],

    df["full_image_path"].iloc[11]
]

result = predict_multiple_image_risk(image_paths)

result

{'number_of_images': 3,
 'image_level_results': [{'predicted_cancer_class': 'Benign',
   'predicted_density': 'D',
   'predicted_birads': 2,
   'cnn_risk_score': 42.23,
   'density_risk_score': 100,
   'birads_risk_score': 25,
   'future_risk_score': 46.59,
   'risk_level': 'Medium Risk',
   'feedback': 'Some areas may need further review. A follow-up consultation with a healthcare professional is recommended.',
   'image_path': '/content/Preprocessed_Dataset/inbreast/inbreast_1.jpg'},
  {'predicted_cancer_class': 'Benign',
   'predicted_density': 'D',
   'predicted_birads': 2,
   'cnn_risk_score': 51.37,
   'density_risk_score': 100,
   'birads_risk_score': 25,
   'future_risk_score': 52.07,
   'risk_level': 'Medium Risk',
   'feedback': 'Some areas may need further review. A follow-up consultation with a healthcare professional is recommended.',
   'image_path': '/content/Preprocessed_Dataset/inbreast/inbreast_2.jpg'},
  {'predicted_cancer_class': 'Benign',
   'predicted_density': 'C